# Wrapper Methods for Obesity Classification – Practice Skeleton

**Short name (GitHub):** `WrapFS_Obes`  
**Lab source:** Codecademy *Wrapper Methods* (UCI obesity eating-habits survey)  
**Language:** Python (pandas + scikit-learn + mlxtend)

Use this notebook to practice. Open **`WrapFS_Obes_Solution.ipynb`** only after you attempt each exercise.  
Companion files: `WrapFS_Obes_Cheatsheet.docx`, `WrapFS_Obes_Reusable_Template.ipynb`, `wrapfs_obes_flowchart.png`, `WrapFS_Obes.py`.

### Learning objectives
- Fit a logistic regression on 18 survey items and record **in-sample accuracy**
- Run **sequential forward selection (SFS)** and read `subsets_[k]`
- Run **sequential backward selection (SBS)** (and the floating variant)
- Run **recursive feature elimination (RFE)** after standardizing
- Compare subset accuracy with the 18-feature baseline
- Alternate implementations (`sklearn.feature_selection.SequentialFeatureSelector`, L1-logistic, mutual information)
- Extra practice: hold-out split, drop collinear transport dummies, `k` sweep
- Monte-Carlo: subset size, sample size, label noise, extra junk columns
- Rewrite the same result for an epidemiologist, an analyst, a public-health director, a non-specialist

### Data files
- `data/obesity.csv` — 2,111 respondents × 18 encoded predictors + `NObeyesdad` (1 = obese)

### Flowchart
Open `wrapfs_obes_flowchart.png` while you work.


## Inline cheat-sheet (keep this cell visible)

See also **`WrapFS_Obes_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Split | `X = df.drop(columns=["NObeyesdad"]); y = df["NObeyesdad"]` |
| Baseline LR | `LogisticRegression(max_iter=1000).fit(X,y).score(X,y)` |
| SFS | `SFS(lr, k_features=9, forward=True, floating=False, scoring="accuracy", cv=0)` |
| SBS | same but `forward=False`, `k_features=7` |
| Floating | `floating=True` lets the search add *and* drop at each step (SFFS / SBFS) |
| Inspect | `sfs.subsets_[k]["feature_names"]`, `sfs.subsets_[k]["avg_score"]` |
| Plot | `plot_sfs(sfs.get_metric_dict()); plt.show()` |
| Scale for RFE | `StandardScaler().fit_transform(X)` so `|coef|` ranks are fair |
| RFE | `RFE(estimator=lr, n_features_to_select=8).fit(Xs, y)` |
| Kept names | `[f for f, ok in zip(features, rfe.support_) if ok]` |
| `cv=0` | **in-sample** score (lesson default). Prefer `cv=5` in production. |

**Flow:** load → baseline LR → SFS → SBS → scale → RFE → compare → practice → simulate.

**Published lesson numbers (this file, `cv=0`):** all-18 ≈ **0.766**; SFS-9 ≈ **0.784**; SFS-6 ≈ **0.769**; SBS-7 ≈ **0.764**; RFE-6 ≈ **0.758**; RFE-8 ≈ **0.768**.


## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE, SequentialFeatureSelector as SkSFS
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

%matplotlib inline
np.set_printoptions(precision=4, suppress=True)
print("Libraries loaded")


## 1. Load and inspect

The table comes from Palechor & de la Hoz (UCI). Categorical answers were already encoded as numbers.

* `Gender` 1 = male, 0 = female
* `Age` years
* `family_history_with_overweight` 1 / 0
* `FAVC` frequent high-calorie food 1 / 0
* `FCVC` vegetables in meals (higher = more)
* `NCP` number of main meals
* `CAEC` food between meals (0–3)
* `SMOKE` 1 / 0
* `CH2O` water intake (0–2)
* `SCC` monitors calories 1 / 0
* `FAF` physical activity (0–3)
* `TUE` screen time (0–2)
* `CALC` alcohol (0–3)
* `Automobile`, `Bike`, `Motorbike`, `Public_Transportation`, `Walking` — one-hot primary transport
* `NObeyesdad` **1 = obese**, 0 = not

### Task 1.1
Load `data/obesity.csv`, print `.shape`, `.head()`, class balance, and dtypes.


In [ ]:
# YOUR CODE HERE
obesity = pd.read_csv("data/obesity.csv")
# inspect shape, head, value_counts of NObeyesdad


### Task 1.2 — split `X` and `y`

Drop the outcome column into `X` and keep `NObeyesdad` as `y`.


In [ ]:
# YOUR CODE HERE
X = None
y = None
print("X shape", getattr(X, "shape", None), "y mean (obesity rate)", None)


## 2. Baseline logistic regression

### Task 2.1
Create `lr = LogisticRegression(max_iter=1000)`, fit on `X, y`, print `lr.score(X, y)`.

Write the number down. Every wrapper will be judged against it.


In [ ]:
# YOUR CODE HERE
lr = None


## 3. Sequential Forward Selection

Start from the empty set. At each step add the feature that most improves `scoring`.

### Task 3.1
Build `sfs` with `estimator=lr`, `k_features=9`, `forward=True`, `floating=False`, `scoring="accuracy"`, `cv=0`. Fit it (about 30 s).


In [ ]:
# YOUR CODE HERE
sfs = None


### Task 3.2
Print `sfs.subsets_[9]`. Then print the chosen names and `avg_score`. Does 9 features beat all 18?


In [ ]:
# YOUR CODE HERE


### Task 3.3
`plot_sfs(sfs.get_metric_dict())` and show the figure.


In [ ]:
# YOUR CODE HERE


### Task 3.4 — published alternate (`k_features=6`)

The official solution notebook used **6** features, not 9. Repeat SFS with `k_features=6` and compare names + score.


In [ ]:
# YOUR CODE HERE
sfs6 = None


## 4. Sequential Backward Selection

Start from the full set. Drop the feature whose removal hurts accuracy the least.

### Task 4.1
`sbs` with `k_features=7`, `forward=False`, `floating=False`, `scoring="accuracy"`, `cv=0`. Fit.


In [ ]:
# YOUR CODE HERE
sbs = None


### Task 4.2
Print `sbs.subsets_[7]`, the names, and `avg_score`. Plot with `plot_sfs`.


In [ ]:
# YOUR CODE HERE


### Task 4.3 — floating alternate (SBFS)

Set `floating=True` (sequential *backward floating* selection). Does the 7-set change?


In [ ]:
# YOUR CODE HERE
sbfs = None


## 5. Recursive Feature Elimination

RFE repeatedly fits the estimator and drops the weakest `|coef|`. **Scale first** so Age (years) is comparable to 0/1 flags.

### Task 5.1
`features = X.columns` (save names **before** you overwrite `X`).


In [ ]:
# YOUR CODE HERE
features = None


### Task 5.2
Standardize `X` into a DataFrame (keep column names if you can).


In [ ]:
# YOUR CODE HERE
# X_scaled = pd.DataFrame(StandardScaler().fit_transform(X), columns=features)


### Task 5.3
The starter text asks for **8** features; the published solution used **6**. Fit both.

`rfe = RFE(estimator=lr, n_features_to_select=8)` then `.fit`. Build `rfe_features` with the list comprehension

```
[f for (f, support) in zip(features, rfe.support_) if support]
```

Print names, `rfe.ranking_`, and `rfe.score(...)`.


In [ ]:
# YOUR CODE HERE
rfe = None
rfe6 = None


## 6. Compare the three wrappers

### Task 6.1
Fill a small table (print a DataFrame) with method, *k*, accuracy, and feature list. Which features appear in every subset?


In [ ]:
# YOUR CODE HERE


## 7. Alternate code (same scientific question)

### Task 7.1 — sklearn `SequentialFeatureSelector`

`from sklearn.feature_selection import SequentialFeatureSelector as SkSFS`  
`direction="forward"` or `"backward"`, `n_features_to_select=6`, `cv=None` to mimic `cv=0`.


In [ ]:
# YOUR CODE HERE


### Task 7.2 — L1-logistic as an *embedded* method

`LogisticRegression(penalty="l1", solver="liblinear", C=0.08, max_iter=1000)` on scaled `X`. Features with `coef_ == 0` were shrunk out.


In [ ]:
# YOUR CODE HERE


### Task 7.3 — filter baseline (mutual information)

`mutual_info_classif(X, y)` and take the top 6. Wrappers usually beat a pure filter because they see the *model*.


In [ ]:
# YOUR CODE HERE


## 8. More practice

### Task 8.1 — hold-out accuracy (the honest number)

`cv=0` scores the *same rows the model was fit on*. Split 70/30, stratify, refit LR on the SFS-9 columns vs all 18, compare **test** accuracy.


In [ ]:
# YOUR CODE HERE


### Task 8.2 — transport dummies are a simplex

The five transport columns sum to 1. Drop `Public_Transportation` as the reference, rerun SFS-6. Do names stabilize?


In [ ]:
# YOUR CODE HERE


### Task 8.3 — sweep `k` for SFS

For `k` in 2..12 record `avg_score`. Where does the curve flatten? That is the smallest survey you can defend.


In [ ]:
# YOUR CODE HERE


## 9. Simulation (edit the knobs)

Change `K_TARGET`, `N_SUB`, `FLIP_P`, `N_JUNK`, `N_REPS` and re-run. Each rep draws a subsample, optionally flips labels, optionally glues Gaussian junk columns, then records SFS-k in-sample accuracy and a hold-out score.


In [ ]:
# --- knobs ---
K_TARGET = 6
N_SUB = 800          # None = use all 2111 rows
FLIP_P = 0.00        # label-noise rate
N_JUNK = 0           # extra N(0,1) columns
N_REPS = 8
SEED = 7
# -------------

rng = np.random.default_rng(SEED)
# YOUR CODE HERE: loop N_REPS, print mean/std of in-sample and hold-out acc


## 10. Audience rewrite

Using the attached audience PDFs, write **four 4–6 line notes** for the same finding (SFS-9 ≈ 0.784 vs all-18 ≈ 0.766; core items = family history, FAVC, CAEC, SCC, Age):

1. Epidemiologist / statistician (expert)  
2. Survey / data analyst (technician)  
3. Public-health director (executive)  
4. General reader (nonspecialist)

Do **not** claim a clinical diagnosis. This is a teaching survey model.


In [ ]:
# Write your four notes here as strings, then print them.
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""


## 11. What this model can and cannot do

**Fits:** shrinking a long lifestyle questionnaire; ranking which items a clinic should keep; teaching wrapper vs filter vs embedded selection.

**Does not fit:** diagnosing obesity from 9 answers; replacing BMI; causal claims (“snacking *causes* obesity”); deployment without a proper train/validation/test split and calibration.

Open `WrapFS_Obes_Solution.ipynb` to check your numbers.
